# Final Conclusion – Notebook 01

## Data Exploration

In this notebook, we explored the COCO 2017 dataset
and examined its textual and visual annotations.

### Main Results

- 118,287 images in the original COCO training annotations.
- 591,753 human-written caption annotations.
- Approximately five captions per image.
- 860,001 object annotations.
- 80 object categories.
- 10,000 local Training images.
- 100 separate Validation images.

### Importance for Our Project

The exploration confirmed that COCO provides two important
sources of information:

1. **Human Captions:** Natural-language descriptions of images.
2. **Object Annotations:** Structured information about objects
   present in images.

These sources will support the development of our
Metadata-Filtered RAG for Image Captioning system.

### Next Step

The next notebook will focus on Data Cleaning:
detecting corrupted images, identifying duplicates,
checking missing files, and preparing clean metadata.

The Training, Validation, and Test datasets will remain
separate throughout the project.

## Step 13: Verify the Dataset Structure

### Description
Before continuing with preprocessing, we verify that the local
dataset contains the expected number of images.

### Objective
The objective is to check the number of images in each dataset
and confirm that the folder organization is correct.

### Method
We count all supported image files in the Training,
Validation, and Test directories.

In [24]:
# Verify Training and Validation image counts

train_count = count_images(TRAIN_DIR)
validation_count = count_images(VALIDATION_DIR)

dataset_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation"],
    "Expected Images": [10000, 100],
    "Actual Images": [train_count, validation_count]
})

dataset_summary["Status"] = dataset_summary.apply(
    lambda row: "Correct"
    if row["Expected Images"] == row["Actual Images"]
    else "Check Required",
    axis=1
)

display(dataset_summary)

,Dataset,Expected Images,Actual Images,Status
0,Training,10000,10000,Correct
1,Validation,100,100,Correct


## Step 12: Dataset Organization

### Description
Our project uses a subset of the original COCO 2017 dataset.
The images are organized into three separate datasets:
Training, Validation, and Test.

### Dataset Structure

| Dataset | Number of Images | Purpose |
|---------|------------------|---------|
| Training | 10,000 | Main dataset used for development and retrieval preparation |
| Validation | 100 | Used to validate and evaluate intermediate results |
| Test | 100 | Reserved for final evaluation |

### Data Separation
The Training and Validation datasets are stored in separate folders.
The Test dataset is also kept separate and will not be included
in the retrieval knowledge base.

### Objective
This organization ensures that each dataset has a specific role
and helps prevent data leakage during the experiments.

## Step 11: Data Exploration Summary

### Introduction
This section summarizes the main findings obtained during
the exploration of the COCO dataset.

### Main Findings
- The original COCO dataset contains 118,287 images.
- It contains 591,753 human-written captions.
- Each image has approximately five captions.
- The instances dataset contains 860,001 object annotations.
- The dataset includes 80 object categories.
- Our local project contains 10,000 training images.
- Our local validation folder contains 100 images.

### Conclusion
The initial exploration confirms that the COCO dataset provides
both textual descriptions and structured object annotations.

These two types of information are important for our project,
Metadata-Filtered RAG for Image Captioning.

The captions provide natural-language descriptions, while object
annotations provide structured metadata about image content.

The dataset is now ready for the next exploration and preprocessing
tasks, including data cleaning, statistical analysis, and split verification.

## Step 10: Compare the Original Dataset with Local Images

### Description
The original COCO annotation files contain information about
118,287 images. However, our project uses a smaller subset.

### Objective
The objective is to compare the original dataset size with
the number of images available in our local training and
validation folders.

### Method
We count the local image files and compare these counts
with the original COCO annotation statistics.

In [23]:
# Count local image files

def count_images(folder):
    extensions = ["*.jpg", "*.jpeg", "*.png"]
    return sum(
        len(list(folder.glob(ext)))
        for ext in extensions
    )

train_count = count_images(TRAIN_DIR)
validation_count = count_images(VALIDATION_DIR)

summary = {
    "Original COCO images": len(captions_data["images"]),
    "Original caption annotations": len(captions_data["annotations"]),
    "Original object annotations": len(instances_data["annotations"]),
    "Object categories": len(instances_data["categories"]),
    "Local training images": train_count,
    "Local validation images": validation_count
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["Description", "Value"]
)

display(summary_df)

,Description,Value
0,Original COCO images,118287
1,Original caption annotations,591753
2,Original object annotations,860001
3,Object categories,80
4,Local training images,10000
5,Local validation images,100


## Step 9. Explore Object Annotations and Categories

### Description
The COCO instances file contains object annotations that describe the
objects present in images. Each object is associated with a category.

### Objective
The objective is to discover the number of object annotations and
the different object categories available in the dataset.

### Method
We inspect the JSON structure, count the annotations and categories,
and display some examples of object categories.

In [22]:
# Inspect the structure of the instances dataset

print("Keys in instances file:")
print(instances_data.keys())

print("\nNumber of object annotations:")
print(len(instances_data["annotations"]))

print("\nNumber of object categories:")
print(len(instances_data["categories"]))

print("\nFirst 10 object categories:")

categories_df = pd.DataFrame(instances_data["categories"])
display(categories_df.head(10))

Keys in instances file:
dict_keys(['info', 'licenses', 'images', 'annotations', 'categories'])

Number of object annotations:
860001

Number of object categories:
80

First 10 object categories:


,supercategory,id,name
0,person,1,person
1,vehicle,2,bicycle
2,vehicle,3,car
3,vehicle,4,motorcycle
4,vehicle,5,airplane
5,vehicle,6,bus
6,vehicle,7,train
7,vehicle,8,truck
8,vehicle,9,boat
9,outdoor,10,traffic light


### Conclusion

The COCO instances dataset contains 860,001 object annotations
distributed across 80 object categories.

These categories represent different objects such as people,
vehicles, animals, food, furniture, and household objects.

This information will help us describe image content using
structured metadata and support metadata-based image retrieval.

## Step 8. Load the COCO Object Annotations

### Description
The COCO instances annotation file contains information about objects
detected and labeled in images, including their categories and bounding boxes.

### Objective
The objective is to explore the object annotations and understand
the categories available in the dataset.

### Method
We load the `instances_train2017.json` file and inspect its structure.

In [20]:
# Load the object annotations JSON file

with open(INSTANCES_PATH, "r", encoding="utf-8") as file:
    instances_data = json.load(file)

print("Instances annotation file loaded successfully!")

Instances annotation file loaded successfully!


## Step 7.Analyze the Number of Captions per Image

### Description
In this step, we analyze how many human-written captions are associated with each image.

### Objective
The objective is to understand the distribution of captions per image and verify whether the dataset follows the expected COCO structure.

### Method
We count the number of annotations associated with each image using the `image_id` field.

### Expected Result
Most images should have approximately five captions.

In [19]:
# Count the number of captions associated with each image

caption_counts = pd.Series(
    [item["image_id"] for item in captions_data["annotations"]]
).value_counts()

# Display statistical information
print("Caption statistics per image:")
display(caption_counts.describe())

Caption statistics per image:


count    118287.000000
mean          5.002688
std           0.052268
min           5.000000
25%           5.000000
50%           5.000000
75%           5.000000
max           7.000000
Name: count, dtype: float64

### Conclusion

The analysis shows that the COCO training captions dataset contains
118,287 images and 591,753 caption annotations.

Each image has at least five captions. The average number of captions
per image is approximately 5.0027, with a minimum of 5 and a maximum
of 7 captions.

These results confirm that the dataset provides multiple human-written
descriptions for each image, which will be useful for image captioning
and retrieval tasks.

## Step 6.Inspect One Image and One Caption
Objective:

Understand the structure of an image record and a caption annotation.

In [17]:
print("First image:")
print(captions_data["images"][0])

print("\nFirst caption:")
print(captions_data["annotations"][0])

First image:
{'license': 3, 'file_name': '000000391895.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000391895.jpg', 'height': 360, 'width': 640, 'date_captured': '2013-11-14 11:18:45', 'flickr_url': 'http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg', 'id': 391895}

First caption:
{'image_id': 203564, 'id': 37, 'caption': 'A bicycle replica with a clock as the front wheel.'}


## Step 5.Inspect Caption Dataset Structure
Description:

We inspect the main keys of the JSON file to understand how COCO organizes image information and caption annotations.

In [16]:
print("Keys in captions file:")
print(captions_data.keys())

print("\nNumber of images:")
print(len(captions_data["images"]))

print("\nNumber of captions:")
print(len(captions_data["annotations"]))

Keys in captions file:
dict_keys(['info', 'licenses', 'images', 'annotations'])

Number of images:
118287

Number of captions:
591753


## Step 4.Load COCO Caption Annotations
Objective:

Load the caption annotation file to access image information and human-written descriptions

In [15]:
with open(CAPTIONS_PATH, "r", encoding="utf-8") as file:
    captions_data = json.load(file)

print("Captions annotation file loaded successfully!")

Captions annotation file loaded successfully!


## Step 3.Explore Annotation Files
Introduction:

The COCO dataset provides two main annotation files. We first inspect their availability and names before loading their contents.

In [14]:
print("Files in annotations folder:")

for file in ANNOTATIONS_DIR.iterdir():
    print(file.name)

Files in annotations folder:
captions_train2017.json
instances_train2017.json


## Step 2.Define Project Paths
Introduction:

In this step, we define the paths to the project folders and annotation files. This allows us to access the dataset without repeatedly writing long file paths.

In [13]:
PROJECT_DIR = Path(r"C:\Users\PC\Desktop\Metadata_RAG_START")

ANNOTATIONS_DIR = PROJECT_DIR / "annotations"
IMAGES_DIR = PROJECT_DIR / "images"

TRAIN_DIR = IMAGES_DIR / "train_images"
VALIDATION_DIR = IMAGES_DIR / "validation_images"

DATA_DIR = PROJECT_DIR / "data"
VISUALIZATIONS_DIR = PROJECT_DIR / "visualizations"

CAPTIONS_PATH = ANNOTATIONS_DIR / "captions_train2017.json"
INSTANCES_PATH = ANNOTATIONS_DIR / "instances_train2017.json"

print("Project exists:", PROJECT_DIR.exists())
print("Annotations folder exists:", ANNOTATIONS_DIR.exists())
print("Train images folder exists:", TRAIN_DIR.exists())
print("Validation images folder exists:", VALIDATION_DIR.exists())

Project exists: True
Annotations folder exists: True
Train images folder exists: True
Validation images folder exists: True


## Step 1.Import Libraries
أولاً: واش معناها؟

قبل ما نبدأو تحليل البيانات، لازم نستوردو المكتبات اللي تساعدنا:
- json: لقراءة ملفات COCO بصيغة JSON.
- pandas: لتنظيم البيانات وتحليلها في جداول.
- matplotlib: لإنشاء الرسوم البيانية.
- PIL: لفتح الصور والتحقق منها.
- Path: للتعامل مع مسارات الملفات والمجلدات.

In [12]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

print("Libraries imported successfully!")

Libraries imported successfully!



# 01 — Data Exploration

## 1. Introduction

This notebook presents the initial exploration of the MS COCO dataset used in the project "Metadata-Filtered RAG for Image Captioning".

The purpose of this stage is to understand the dataset structure, examine the available images and annotations, and identify the information needed for the next preprocessing stages.

## 2. Objectives

The main objectives are:

- Understand the structure of the COCO dataset.
- Explore image annotation files.
- Examine human-written captions.
- Explore object categories and annotations.
- Verify the relationship between image IDs and captions.

## 3. Dataset Description

The project uses two main annotation files:

- `captions_train2017.json`: contains image information and human-written captions.
- `instances_train2017.json`: contains object annotations and category information.

The images are organized into two folders:

- `images/train_images/`
- `images/validation_images/`

## 4. Methodology

The exploration is performed using Python libraries such as Pandas, Matplotlib, JSON, and Pillow.

We begin by loading the annotation files, inspecting their structure, and checking the relationship between local image files and their corresponding annotations.
